# Tutorial 3: Wrapping an unmodified Gymnasium environment

Previous tutorials defined custom `gym.Env` subclasses. This one uses `gymnasium.MountainCarContinuous-v0` **without any source modifications** — imported directly from `gymnasium` and wrapped with `zrth.gym.Env`.

The analyzer handles the patterns that show up in real off-the-shelf gym envs — array state, helper calls, augmented assignment, and so on. Anything it can't translate symbolically is left as an opaque term, with the real env producing the value at runtime via `self._backing_env`.

## Wrapping the env

Make the environment and pass it to `Env`. We also create explicit `obs_wire` / `act_wire` pairs and attach them up front — same effect as letting the wrapper allocate them, but having handles to the wires lets us wire a controller into them later and compose a closed loop. Outer wrappers from `gym.make` (`TimeLimit`, `OrderEnforcing`, ...) keep doing their job at runtime; the analyzer peels them off internally to extract the symbolic module from the raw env class.

In [1]:
import numpy as np
import gymnasium as gym
from zrth import Wire, Float
from zrth.gym import Env

obs_wire = (Wire(Float(2)), Wire(Float(2)))
act_wire = (Wire(Float(1)), Wire(Float(1)))

env = Env(gym.make('MountainCarContinuous-v0', render_mode='rgb_array'),
          observation=obs_wire, action=act_wire)
print(env)

module
  external
    w2, w3 : Float(1)
  interface
    w0, w1 : Float(2)
    w14, w15 : Float(1)
    w16, w17 : Bool(1)
    w18, w19 : Bool(1)
  private
    w4, w5 : Float(2)
  atom controls w1, w5, w15, w17, w19 reads w2, w4
  init
    Tensor([-1]) w6; 
    Tensor([1]) w7; 
    Tensor([-1.2000000476837158]) w8; 
    Tensor([0.6000000238418579]) w9; 
    Tensor([0.07000000029802322]) w10; 
    Tensor([0.44999998807907104]) w11; 
    Const: 0 w12; 
    Tensor([0.001500000013038516]) w13; 
    utils.maybe_parse_reset_bounds(...) w20; 
    utils.maybe_parse_reset_bounds(...) w21; 
    np_random.uniform(...) w22; 
    Tensor([0]) w23; 
    Stack w24; w22, w23
    Id w5; w24
    Id w1; w24
    Tensor([0]) w15; 
    Const: false w17; 
    Const: false w19; 
  update
    Tensor([-1]) w6; 
    Tensor([1]) w7; 
    Tensor([-1.2000000476837158]) w8; 
    Tensor([0.6000000238418579]) w9; 
    Tensor([0.07000000029802322]) w10; 
    Tensor([0.44999998807907104]) w11; 
    Const: 0 w12; 
    Tenso

## Reading the output

- **external**: the action wire (1-D continuous force)
- **interface**: observation `Float(2)` (position, velocity), reward, terminated, truncated
- **private**: internal `self.state` — a 2-element array, stored as a single `Float(2)` wire pair
- **init / update**: the extracted `reset` and `step` bodies.
  - `utils.maybe_parse_reset_bounds(...)` and `np_random.uniform(...)` appear as **uninterpreted** terms: the analyzer couldn't trace them symbolically, so they're left opaque and filled in by the real env at runtime.
  - The force/gravity/clipping math from `step` is fully symbolic (`Mul`, `Cos`, `Add`, `Ite`, ...).

## Training a small policy

We train a tiny `nn.Module` by **imitation** of an "energy-pumping" controller: push in the direction of velocity, $a = \mathrm{sign}(\dot x)$. Training samples random states uniformly from the env's bounds and supervises against the energy-pumping target (smoothed with `tanh` for a usable gradient).

In [2]:
import torch
import torch.nn as nn

class Policy(nn.Module):
    """(position, velocity) -> force in [-1, 1]."""
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 16)
        self.fc2 = nn.Linear(16, 1)
    def forward(self, obs):
        return torch.tanh(self.fc2(torch.relu(self.fc1(obs))))

torch.manual_seed(0)
policy = Policy()
opt = torch.optim.Adam(policy.parameters(), lr=0.01)

for epoch in range(300):
    pos = torch.empty(128, 1).uniform_(-1.2, 0.6)
    vel = torch.empty(128, 1).uniform_(-0.07, 0.07)
    obs = torch.cat([pos, vel], dim=1)
    target = torch.tanh(vel * 100.0)            # smooth sign(velocity)
    loss = ((policy(obs) - target) ** 2).mean()
    opt.zero_grad(); loss.backward(); opt.step()
    if epoch % 50 == 0:
        print(f"epoch {epoch:3d}  loss = {loss.item():.4f}")

epoch   0  loss = 0.8514
epoch  50  loss = 0.7844
epoch 100  loss = 0.4759
epoch 150  loss = 0.2085
epoch 200  loss = 0.1020
epoch 250  loss = 0.0593


## Composing a closed loop

Wrap the trained `policy` as a `zrth.torch.Module` on the same `obs_wire` / `act_wire` pairs and compose it with `env`. The resulting module wires the controller's output into the env's action.

In [3]:
from zrth.torch import Module

wrapped_policy = Module(policy, extl=obs_wire, intf=act_wire)
closed_loop = Env(env, wrapped_policy)
print(closed_loop)

module
  interface
    w0, w1 : Float(2)
    w14, w15 : Float(1)
    w16, w17 : Bool(1)
    w18, w19 : Bool(1)
    w2, w3 : Float(1)
  private
    w4, w5 : Float(2)
  atom controls w1, w5, w15, w17, w19 reads w2, w4
  init
    Tensor([-1]) w6; 
    Tensor([1]) w7; 
    Tensor([-1.2000000476837158]) w8; 
    Tensor([0.6000000238418579]) w9; 
    Tensor([0.07000000029802322]) w10; 
    Tensor([0.44999998807907104]) w11; 
    Const: 0 w12; 
    Tensor([0.001500000013038516]) w13; 
    utils.maybe_parse_reset_bounds(...) w20; 
    utils.maybe_parse_reset_bounds(...) w21; 
    np_random.uniform(...) w22; 
    Tensor([0]) w23; 
    Stack w24; w22, w23
    Id w5; w24
    Id w1; w24
    Tensor([0]) w15; 
    Const: false w17; 
    Const: false w19; 
  update
    Tensor([-1]) w6; 
    Tensor([1]) w7; 
    Tensor([-1.2000000476837158]) w8; 
    Tensor([0.6000000238418579]) w9; 
    Tensor([0.07000000029802322]) w10; 
    Tensor([0.44999998807907104]) w11; 
    Const: 0 w12; 
    Tensor([0.001500

## Rollout with rendering

Run `closed_loop` like any `gym.Env`. The action argument to `step` is ignored — the composed controller's output drives the env. We collect RGB frames as we go.

In [4]:
closed_loop.reset(seed=42)
frames = [closed_loop.render()]

with torch.no_grad():
    for t in range(500):
        obs, r, term, trunc, _ = closed_loop.step(0)   # action ignored — controller drives
        frames.append(closed_loop.render())
        if term or trunc:
            break
print(f"reached the goal in {t + 1} steps  (final position {obs[0]:.3f})")

reached the goal in 151 steps  (final position 0.504)


Animate the collected frames inline.

In [5]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

fig, ax = plt.subplots(figsize=(6, 4))
img = ax.imshow(frames[0])
ax.axis('off')
anim = animation.FuncAnimation(
    fig, lambda i: [img.set_data(frames[i]) or img], frames=len(frames),
    interval=50, blit=True,
)
plt.close(fig)
HTML(anim.to_jshtml())

## Interactive viewer

Launch the interactive module viewer on the composed `closed_loop` to see the env and policy atoms wired together.

In [6]:
from zrth.visual import show
stop = show(closed_loop)

Module viewer running at http://127.0.0.1:57822 (ws://127.0.0.1:57823)
